# C4 — Feature ablation and class imbalance (exploratory)

L1: feature groups CVSS / description text / point-in-time EPSS — each alone, each pair, all — with the C1-R-selected XGBoost (10 seeds) and LR, **not re-tuned per subset**. Imbalance on XGBoost: none, `scale_pos_weight` √(neg/pos) and neg/pos, undersampling to 1:20; LR balanced. L3 (**simulated, reported separately**): finding-only vs context-only vs all. Design: Study C addendum 2 (commit 59bd4b1). No hypothesis registered.

Reads saved outputs; run: `python -m src.models.c4_ablation --stage l1` and `--stage l3` (~45 min).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

## L1 — ablation and imbalance (10 seeds, mean ± std)

In [2]:
pd.read_csv("outputs/c4_l1_summary.csv", index_col=[0, 1, 2])

pr_auc precision_at_100      ndcg_at_100          roc_auc            brier              ece
kind      variant           model                                                                                                        
ablation  all               logreg      0.0235 ± 0.0       0.12 ± 0.0     0.1404 ± 0.0     0.8344 ± 0.0     0.0013 ± 0.0     0.0003 ± 0.0
imbalance spw_full          xgboost   0.023 ± 0.0007    0.145 ± 0.005  0.1536 ± 0.0064  0.8317 ± 0.0018   0.0637 ± 0.001  0.2013 ± 0.0021
ablation  cvss+epss         logreg      0.0227 ± 0.0       0.13 ± 0.0     0.1424 ± 0.0     0.7745 ± 0.0     0.0014 ± 0.0     0.0002 ± 0.0
imbalance balanced          logreg      0.0227 ± 0.0       0.12 ± 0.0      0.139 ± 0.0     0.8374 ± 0.0     0.0525 ± 0.0     0.1425 ± 0.0
          spw_sqrt          xgboost  0.0225 ± 0.0005       0.13 ± 0.0  0.1423 ± 0.0032  0.8277 ± 0.0015     0.0029 ± 0.0  0.0255 ± 0.0004
ablation  all               xgboost   0.022 ± 0.0002    0.132 ± 0.006  0.1394 ± 0.0046  0.8274 ± 0.0023     0.0014 ± 0.0     0.0007 ± 0.0
imbalance undersample_1to20 xgboost  0.0204 ± 0.0011   0.132 ± 0.0117   0.1403 ± 0.009  0.8256 ± 0.0019  0.0026 ± 0.0001  0.0203 ± 0.0005
ablation  cvss+epss         xgboost  0.0203 ± 0.0003   0.146 ± 0.0111   0.1474 ± 0.008   0.791 ± 0.0031     0.0014 ± 0.0     0.0003 ± 0.0
          epss              logreg      0.0193 ± 0.0       0.11 ± 0.0     0.1262 ± 0.0     0.6889 ± 0.0     0.0014 ± 0.0     0.0004 ± 0.0
          text+epss         logreg      0.0186 ± 0.0       0.11 ± 0.0      0.139 ± 0.0     0.7965 ± 0.0     0.0014 ± 0.0     0.0005 ± 0.0
                            xgboost   0.017 ± 0.0012   0.126 ± 0.0092  0.1438 ± 0.0138  0.7656 ± 0.0034     0.0014 ± 0.0     0.0003 ± 0.0
          epss              xgboost  0.0151 ± 0.0004   0.128 ± 0.0125   0.1381 ± 0.009  0.6648 ± 0.0008     0.0014 ± 0.0     0.0005 ± 0.0
          cvss+text         logreg      0.0073 ± 0.0       0.02 ± 0.0     0.0176 ± 0.0     0.8374 ± 0.0     0.0014 ± 0.0     0.0035 ± 0.0
                            xgboost  0.0072 ± 0.0002       0.01 ± 0.0  0.0085 ± 0.0006  0.8181 ± 0.0017     0.0015 ± 0.0  0.0051 ± 0.0001
          cvss              logreg      0.0058 ± 0.0       0.02 ± 0.0     0.0259 ± 0.0     0.7792 ± 0.0     0.0014 ± 0.0     0.0054 ± 0.0
          text              xgboost  0.0057 ± 0.0001    0.018 ± 0.004  0.0176 ± 0.0055  0.7696 ± 0.0014     0.0014 ± 0.0      0.003 ± 0.0
                            logreg      0.0049 ± 0.0        0.0 ± 0.0        0.0 ± 0.0      0.791 ± 0.0     0.0014 ± 0.0     0.0025 ± 0.0
          cvss              xgboost  0.0044 ± 0.0002    0.002 ± 0.004  0.0056 ± 0.0143  0.7698 ± 0.0016     0.0014 ± 0.0  0.0063 ± 0.0001

## Each subset / imbalance variant vs the full model (paired bootstrap)

In [3]:
pd.read_csv("outputs/c4_l1_tests.csv", index_col=0).round(4)

,pr_auc_diff,pr_auc_ci_low,pr_auc_ci_high,pr_auc_p_not_better,p_at_100_diff,p_at_100_ci_low,p_at_100_ci_high,p_at_100_p_not_better
ablation_xgboost_cvss_vs_all,-0.0179,-0.0362,-0.0088,1.000,-0.14,-0.2000,-0.06,1.000
ablation_xgboost_text_vs_all,-0.0165,-0.0342,-0.0071,1.000,-0.13,-0.1900,-0.05,1.000
ablation_xgboost_epss_vs_all,-0.0071,-0.0129,-0.0019,0.994,-0.01,-0.0400,0.04,0.566
ablation_xgboost_cvss+text_vs_all,-0.0150,-0.0325,-0.0059,1.000,-0.13,-0.1952,-0.06,1.000
ablation_xgboost_cvss+epss_vs_all,-0.0023,-0.0068,0.0012,0.900,0.00,-0.0300,0.06,0.402
ablation_xgboost_text+epss_vs_all,-0.0058,-0.0126,-0.0000,0.974,-0.01,-0.0300,0.03,0.668
imbalance_xgboost_spw_sqrt_vs_none,0.0003,-0.0018,0.0032,0.406,-0.01,-0.0300,0.03,0.778
imbalance_xgboost_spw_full_vs_none,0.0005,-0.0028,0.0044,0.346,0.00,-0.0152,0.05,0.314
imbalance_xgboost_undersample_1to20_vs_none,-0.0011,-0.0030,0.0008,0.904,0.00,-0.0100,0.03,0.762
ablation_logreg_cvss_vs_all,-0.0178,-0.0362,-0.0081,1.000,-0.10,-0.1800,-0.03,0.998


**Reading (L1).**
- **EPSS carries the signal.** Every set without point-in-time EPSS collapses to CVSS-baseline level (CVSS+text PR-AUC ≈ 0.007, vs 0.0068 for CVSS ordering). CVSS+EPSS is statistically indistinguishable from all features (XGBoost −0.0023 [−0.0068, 0.0012]; LR −0.0008 [−0.0071, 0.0058]).
- **Text adds little once CVSS and EPSS are present**, and the 64-d description embedding is weak on its own (PR-AUC ≈ 0.005).
- **Imbalance handling does not help ranking:** no weighting/undersampling variant changes PR-AUC or P@100 significantly — but it **destroys calibration** (Brier 0.0014 → 0.064 with full `scale_pos_weight`, ECE 0.20). C5 shows Platt scaling repairs this. Conclusion: train unweighted.
- Consistency check: "all / xgboost" reproduces C1-R XGBoost exactly (0.0220 ± 0.0002).

## L3 — SIMULATED estate (target: High/Critical, 10 grouped-by-asset splits)

In [4]:
pd.read_csv("outputs/c4_l3_summary.csv", index_col=[0, 1])

,,pr_auc,precision_at_100,ndcg_at_100,roc_auc,brier,ece
target,feature_set,,,,,,
clean,all,0.9976 ± 0.0016,1.0 ± 0.0,1.0 ± 0.0,0.9998 ± 0.0002,0.0015 ± 0.0004,0.0013 ± 0.0004
noisy,all,0.9976 ± 0.0016,1.0 ± 0.0,1.0 ± 0.0,0.9998 ± 0.0002,0.0015 ± 0.0004,0.0013 ± 0.0004
clean,context_only,0.3884 ± 0.0278,0.707 ± 0.0355,0.7113 ± 0.0416,0.8635 ± 0.0086,0.0421 ± 0.0027,0.0167 ± 0.002
noisy,context_only,0.3884 ± 0.0278,0.707 ± 0.0355,0.7113 ± 0.0416,0.8635 ± 0.0086,0.0421 ± 0.0027,0.0167 ± 0.002
clean,finding_only,0.3882 ± 0.022,0.664 ± 0.0439,0.7109 ± 0.0401,0.8736 ± 0.0106,0.04 ± 0.0033,0.0076 ± 0.0013
noisy,finding_only,0.3882 ± 0.022,0.664 ± 0.0439,0.7109 ± 0.0401,0.8736 ± 0.0106,0.04 ± 0.0033,0.0076 ± 0.0013


**Reading (L3, simulated — not evidence about a real estate).** Finding-only and context-only features each reach PR-AUC ≈ 0.39; together ≈ 0.998, i.e. XGBoost recovers the documented label formula almost exactly once it sees both halves of it. This confirms the formula really needs both the finding and the asset context (neither alone suffices), and it is why L3 performance must never be quoted as a model result. The "clean" and "noisy" targets are identical because the simulator's `noise_level` is 0.0, so the registered noise sensitivity is vacuous here (logged in the Deviations Log).